# 🤖 LangGraph Applications — Presentación Técnica

> **Dos aplicaciones de IA agéntica construidas con LangGraph, RAG y observabilidad con Langfuse**

---

| Aplicación | Dominio | Pipeline |
|---|---|---|
| **Clean Code Reviewer** | Calidad de código C# | 7 nodos · fan-out/fan-in paralelo |
| **XTP Analyzer** | Análisis de programas XTP | 6 nodos · secuencial + PR linking |


---
# Parte 1 — Clean Code Reviewer
---

## 📌 Motivación / Problema a Solucionar

El objetivo de la aplicación es de señalar las malas prácticasde programación mas comunes para proveer recomendaciones a desarrolladores. 

## Motivación

En la industria y en la educación, por las constantes fechas de entregables y cortos lapsos de tiempo, el desarrollo Software se enfoca nada mas en "Que funcione". Sin embargo, esto puede generar deudas técnicas. 

### Causas

- Prisas en el desarrollo
- Falta de requerimientos
- Falta de documentacion *

Una aplicación con pobre calidad en su código con lleva a varias consecuencias.

### Consecuencias

- Relentización en el Desarollo
- Aumento de Bugs
- Fuga de Talento y Frustración

## 🗂️ Estructura del Proyecto — Clean Code Reviewer

```
prueba_langchain/
├── CleanCodeReporter/
│   ├── GraphAgent.py       ← Definición del grafo LangGraph (7 nodos)
│   ├── agent_setup.py      ← Inicialización del agente ReAct + herramientas
│   ├── Agent.py            ← Clase base del agente (ChatOpenAI + ReAct loop)
│   ├── Rag.py              ← Núcleo RAG: PDFProcessor → ChromaDB
│   ├── GithubPRAgent.py    ← Creación de GitHub Issues por hallazgo
│   └── dash_app.py         ← Dashboard Dash (visualización de reportes)
├── Helpers/
│   ├── LangfuseCallbackHandler.py  ← Observabilidad: get_callback() + trace_name_context()
│   ├── JsonFormatterAgent.py       ← Corrector de JSON malformado
│   ├── ScorerAgent.py              ← Calificación del reporte de code smells
│   └── FilePatcher.py              ← Aplicación de unified diffs al archivo fuente
├── RAG/
│   ├── pdf_processor.py    ← Chunking de PDFs por capítulo
│   ├── vector_store.py     ← Abstracción ChromaDB
│   └── rag_config.py       ← Configuración (colección, embeddings)
├── Documentos/             ← PDFs de buenas prácticas (alimentan el RAG)
└── Estructuras/
    └── CodeSmellReport.py  ← TypedDict GraphState
```

## 🏗️ Arquitectura Agéntica — Clean Code Reviewer

El pipeline está construido con **LangGraph `StateGraph`** y ejecuta 7 nodos, incluyendo un **bucle de reintento** ante JSON inválido y una **rama paralela** (fan-out / fan-in) para calificación y parcheo simultáneos.

```mermaid
flowchart TD
    START([INICIO])
    read_file["read_file\nAgent ReAct - ChatOpenAI\ntools: read_local_file, read_github_url, find_documents\nout: raw_response"]
    validate_json{"validate_json\nJsonFormatterAgent\nJSON valido?"}
    extract_report["extract_report\nParser JSON\nout: report"]
    score_report["score_report\nScorerAgent - ChatOpenAI\nout: score_json"]
    patch_file["patch_file\nFilePatcher\nout: patched, patch_diff, finding_patches"]
    merge["merge\nfusiona score_json + patch_data\nout: report"]
    create_pr["create_pr\nGithubPRAgent\nout: pr_urls"]
    END([FIN])

    START --> read_file
    read_file --> validate_json
    validate_json -- valido --> extract_report
    validate_json -- reintentar --> read_file
    extract_report --> score_report
    extract_report --> patch_file
    score_report --> merge
    patch_file --> merge
    merge -- repo detectado --> create_pr
    merge -- sin repo --> END
    create_pr --> END
```

### Descripción de los nodos

| Nodo | Responsabilidad | Clave de estado (output) |
|---|---|---|
| `read_file` | Agente ReAct: lee el archivo y detecta code smells via LLM | `raw_response` |
| `validate_json` | Verifica y corrige el JSON devuelto por el LLM | `report_json`, `valid_json` |
| `extract_report` | Parsea el JSON a dict Python | `report` |
| `score_report` *(paralelo)* | Califica la severidad global del reporte | `score_json` |
| `patch_file` *(paralelo)* | Aplica unified diffs y escribe el archivo corregido | `patched`, `patch_diff`, `finding_patches` |
| `merge` | Combina resultados de las ramas paralelas | `report` (enriquecido) |
| `create_issue` | Abre GitHub Issue con todos los hallazgos | `pr_urls` |

### Patrón fan-out / fan-in (ejecución paralela)

LangGraph permite que desde `extract_report` se disparen **dos ramas simultáneas** que escriben en claves de estado disjuntas. `merge_node` espera a que ambas completen antes de continuar.

```python
# Fan-out: ambas ramas corren en paralelo, escriben claves disjuntas
graph.add_edge("extract_report", "score_report")   # → score_json
graph.add_edge("extract_report", "patch_file")     # → patched, patch_diff, finding_patches

# Fan-in: convergen en merge
graph.add_edge("score_report", "merge")
graph.add_edge("patch_file",   "merge")
```

## 📚 Uso de RAG — Clean Code Reviewer

El agente `read_file` tiene acceso a la herramienta `find_documents`, que consulta una **base de conocimiento vectorial** alimentada con PDFs de buenas prácticas de programación.

```mermaid
flowchart LR
    PDFs["Documentos/\nPDFs de buenas practicas"]
    PDF_PROC["PDFProcessor\nchunks por capitulo"]
    VS["ChromaDB\ncollection: default\nVectorStore"]
    TOOL["Tool: find_documents\nagent_setup.py"]
    AGENT["Agent ReAct\nread_file_node"]
    RAG_REF["ragReference en cada finding\ncita APA del top chunk"]

    PDFs --> PDF_PROC --> VS
    AGENT -- consulta smell --> TOOL --> VS
    VS -- top chunk --> TOOL --> RAG_REF
```

### Flujo RAG paso a paso

1. **Ingesta:** `PDFProcessor` divide cada PDF en *chunks* por capítulo y los almacena en **ChromaDB** (colección `default`).
2. **Consulta:** Cuando el agente detecta un code smell, llama `find_documents(smell_name)` — una herramienta LangChain decorada con `@tool`.
3. **Recuperación:** ChromaDB devuelve el chunk más cercano por similitud coseno.
4. **Citación:** El resultado se formatea como referencia APA y se inyecta en el campo `ragReference` del JSON de hallazgos.

```python
# RagCore.find_documents — Helpers/Rag.py
def find_documents(self, text_to_find: str) -> str:
    results = self.vector_store.search(text_to_find)
    top = results[0]
    source  = top.get("source", "Unknown")
    chapter = top.get("chapter", "")
    apa = f"{source} (n.d.). {chapter}."
    return apa   # ← el agente copia esto verbatim en ragReference
```

## 🔭 Observabilidad con Langfuse — Clean Code Reviewer

Cada ejecución del grafo se instrumenta automáticamente con **Langfuse v4**, agrupando todas las trazas bajo el nombre `CleanCodeReviewer`.

```mermaid
flowchart TD
    A["GraphAgent.run"]
    B["get_callback\nLangfuseCallbackHandler.py"]
    C["trace_name_context CleanCodeReviewer\nOTel propagate_attributes"]
    D["compiled_graph.invoke\nstate + callbacks config"]
    E["Cada ChatOpenAI / ScorerAgent\nget_callback + trace_name_context"]
    F["Langfuse UI\ntrazas bajo CleanCodeReviewer"]

    A --> B --> C --> D --> E --> F
```

### Detalles de implementación

| Concepto | Implementación |
|---|---|
| **Session ID** | Ruta del archivo analizado → hash MD5 → `trace_id` de 32 hex chars |
| **Trace Name** | `trace_name_context("CleanCodeReviewer")` — context manager OTel |
| **Callback** | `langfuse.langchain.CallbackHandler` inyectado en el `config` del grafo |
| **Graceful degradation** | Si las env vars de Langfuse no están → `get_callback()` devuelve `None` |

```python
# GraphAgent.py — punto de entrada
def run(file_path: str) -> dict:
    _cb = get_callback(session_id=file_path)          # None si Langfuse no configurado
    _config = {"callbacks": [_cb]} if _cb else {}
    with trace_name_context("CleanCodeReviewer"):     # propaga nombre de traza vía OTel
        return compiled_graph.invoke(initial_state, config=_config)
```

## 🚀 Demo — Ejecutar el Clean Code Reviewer

In [1]:
# -------------------------------------------------------------------
# Configuración compartida de Logger + dropdown de nivel de log
# Ejecuta esta celda ANTES de cualquier demo.
# -------------------------------------------------------------------
import sys
import os
import logging
import ipywidgets as widgets
from IPython.display import display

sys.path.insert(0, os.path.abspath('..'))

from Helpers.Logger import AgentLogger

# Logger compartido para todas las celdas de demo
log = AgentLogger(name="presentacion", level="INFO")

# --- Dropdown de nivel de log ---
level_dropdown = widgets.Dropdown(
    options=["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"],
    value="INFO",
    description="Log level:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px"),
)

def _on_level_change(change):
    new_level = change["new"]
    log._logger.setLevel(new_level)
    for handler in log._logger.handlers:
        handler.setLevel(new_level)
    log._logger.info("Nivel de log cambiado a: %s", new_level)

level_dropdown.observe(_on_level_change, names="value")

display(level_dropdown)
log._logger.info("Logger listo. Selecciona el nivel de log con el dropdown.")

Dropdown(description='Log level:', index=1, layout=Layout(width='260px'), options=('DEBUG', 'INFO', 'WARNING',…

22:06:56  [INFO]     Logger listo. Selecciona el nivel de log con el dropdown.


In [2]:
# Ejemplo de ejecución del Clean Code Reviewer
# Asegúrate de tener el .env configurado con LLM_API_KEY y LANGFUSE_* antes de ejecutar

from CleanCodeReporter.GraphAgent import run

# Cambia la ruta al archivo C# que quieres analizar
FILE_PATH = "Ejemplos/CodeSmell1.cs"

log._logger.info("Iniciando analisis de: %s", FILE_PATH)
result = run(FILE_PATH)

report = result.get("report", {})
log._logger.info("Archivo analizado   : %s", report.get("fileName"))
log._logger.info("Evaluacion          : %s", report.get("summary", {}).get("overallAssessment"))
log._logger.info("Code smells         : %s", report.get("summary", {}).get("smellsDetected"))

if report.get("scoreReport"):
    sr = report["scoreReport"]
    log._logger.info("Score: %s / 100 — Grado: %s", sr.get("score"), sr.get("grade"))

d:\Repositorios\prueba_langchain\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


22:07:04  [INFO]     Procesando 2 PDF(s)...
22:07:04  [INFO]     Procesando 2 PDF(s)...
22:07:07  [DEBUG]      clean-code.pdf -> 1117 chunks
22:07:07  [DEBUG]      clean-code.pdf -> 1117 chunks
22:07:14  [DEBUG]      DesignPatterns.pdf -> 893 chunks
22:07:14  [DEBUG]      DesignPatterns.pdf -> 893 chunks
22:07:14  [INFO]     VectorStore listo — colección='Documentacion', path=.chroma
22:07:14  [INFO]     VectorStore listo — colección='Documentacion', path=.chroma
22:08:17  [INFO]     Indexados 2010 documentos en la colección 'Documentacion'
22:08:17  [INFO]     Indexados 2010 documentos en la colección 'Documentacion'
22:08:21  [INFO]     Iniciando analisis de: Ejemplos/CodeSmell1.cs
22:08:22  [INFO]     📂 read_file_node — analyzing: Ejemplos/CodeSmell1.cs
22:08:22  [INFO]     ──────────────────────────────────────────────────────────────
22:08:22  [INFO]     AGENTE  |  tools: []
22:08:22  [INFO]     Pregunta: Que Code Smells detectas en este archivo? Ejemplos/CodeSmell1.cs
22:08:22  [

d:\Repositorios\prueba_langchain\CleanCodeReporter\GithubPRAgent.py:442: RuntimeWarning: coroutine 'create_issue_per_finding' was never awaited
  return []


In [3]:
# Visualizar los hallazgos individuales
findings = report.get("findings", [])
log._logger.info("Total de hallazgos: %d", len(findings))

for f in findings:
    loc = f.get("location", {})
    log._logger.info(
        "[#%s] %s — %s",
        f.get("id"), f.get("smell"), f.get("severity"),
    )
    log._logger.debug(
        "  Ubicacion: %s :: %s",
        loc.get("className"), loc.get("methodName"),
    )
    log._logger.debug("  Descripcion: %s", f.get("description"))
    if f.get("ragReference"):
        log._logger.debug("  Ref RAG: %s", f.get("ragReference"))

22:09:06  [INFO]     Total de hallazgos: 4
22:09:06  [INFO]     [#1] Dead Code (codigo muerto fuera de cualquier tipo) — Critical
22:09:06  [DEBUG]      Ubicacion: None :: None
22:09:06  [DEBUG]      Descripcion: Al final del archivo hay una linea huerfana 'var user = new User(name, email);' que se encuentra fuera de cualquier clase o metodo. Es codigo inalcanzable y ademas impediria la compilacion del proyecto.
22:09:06  [DEBUG]      Ref RAG: clean-code (n.d.). 4. [Knuth92]..
22:09:06  [INFO]     [#2] Incomplete/duplicated entity creation (Lazy Class / Inconsistent API) — High
22:09:06  [DEBUG]      Ubicacion: UserRegistrationService :: Register
22:09:06  [DEBUG]      Descripcion: Register usa User.Create(name, email) (factory centralizada) mientras que la linea huerfana usa new User(name, email), y UserManager.Validate nunca es invocada. La creacion del usuario y su validacion no estan unificadas entre las dos clases.
22:09:06  [DEBUG]      Ref RAG: clean-code (n.d.). 6. [Refactoring

---
# Parte 2 — XTP Analyzer
---

## 📌 Motivación / Problema a Solucionar

En la industria de fabricación de microprocesadores, se desarrolla Software para validación en la etapa de `Testing`. Este es el proceso crítico de verificación eléctrica y funcional para descartar chips defectuosos y clasificar el rendimiento de cada unidad

![image](./Imagenes/FlujoManufactura.jpg)

En testing, se utiliza programación de un Software para configurar las diferentes pruebas. Ejemplo conceptual:

```
PINMAP {
    PIN VDD_CORE TYPE POWER;
    PIN GND TYPE GROUND;
    PIN CAM_CLK TYPE INPUT;
    PIN MIPI_DATA TYPE INOUT;
    PIN ALERT_N TYPE OUTPUT;
}

LEVELS {
    V_DD_CORE = 0.95V;
    V_IH = 0.84V;
    V_IL = 0.36V;
    V_OH = 0.96V;
    V_OL = 0.24V;
}

TIMING {
    period_ns = 5.0;
    drive_high_ns = 2.5;
    drive_low_ns = 2.5;
    strobe_ns = 3.1;
}

PARAMETRICS {
    I_DDQ_MAX = 15.0mA;
    I_IH_MAX = 1.0uA;
    I_IL_MAX = 1.0uA;
}

FUNCTIONS {
    VECTOR CAM_CLK MIPI_DATA ALERT_N;
    CYCLE (0, "Z", "H");
    CYCLE (1, "1", "H");
    CYCLE (0, "0", "L");
    CYCLE (1, "1", "H");
    CYCLE (0, "0", "L");
    CYCLE (1, "1", "H");
    CYCLE (0, "0", "L");
    CYCLE (1, "1", "H");
}

BINNING {
    PASS_PRIME -> HB_1, SB_1001;
    PASS_ECO -> HB_1, SB_1003;
    FAIL_DIODE -> HB_2, SB_2001;
    FAIL_LEAKAGE -> HB_3, SB_3001;
    FAIL_TIMING -> HB_4, SB_4001;
}

```

![image](./Imagenes/Tester.jpg)

# Proceso de validación

1. Antes de mandar una actualización, se hacen validaciones de ingeniería.<br>
2. Caso: jutifiación de discrepancia entre un fallo u otro.
3. Cualquier discrepancia tiene que ser entendida por ingeniería. 

In [1]:
import pandas as pd

display(pd.read_csv(r"Programas/Bin2Bin_Matrix.csv"))

,Prog_A \ Prog_B,SB_1001_PassPrime,SB_1003_EcoPass,SB_3001_IDDQ_Fail,SB_4001_TimingFail
0,SB_1001_PassPrime,649,5,146,0
1,SB_1003_EcoPass,0,100,0,0
2,SB_3001_IDDQ_Fail,0,0,50,0
3,SB_4001_TimingFail,0,0,0,50


## Retos
1. Multiples cambios entre 2 revisiones (orden de decenas o hasta centenas).
2. Mucha información.
3. Busqueda manual(PR's, descripcion, reportes)
4. Labor extensa y tediosa.

El siguiente proyecto , presenta una propuesta de como agilizar este proceso, <br>
utilizando una arquitectura multi agéntica para ofrecer una justificación inicial <br>
de los hallazgos durante validación con ayuda de IA. 

## 🗂️ Estructura del Proyecto — XTP Analyzer

```
prueba_langchain/
├── XTPAnalyser/
│   ├── AnalysisGraph.py        ← Pipeline de análisis (6 nodos)
│   ├── graph.py                ← Pipeline de generación (2 nodos)
│   ├── Main.py                 ← Punto de entrada del análisis
│   ├── ProgramGeneration.py    ← Punto de entrada de la generación
│   ├── GitCommitGraph.py       ← Grafo para obtener commits de GitHub
│   ├── dashboard.py            ← Dashboard Dash (visualización de análisis)
│   └── Agents/
│       ├── XTPGitCommitAgent.py         ← Obtiene archivos XTP por commit SHA
│       ├── XTPProgramDiffAgent.py       ← Analiza diferencias entre programas + RAG
│       ├── XTPBin2BinMatrixAgent.py     ← Analiza la matriz Bin2Bin + RAG
│       ├── XTPMismatchJustificationAgent.py  ← Justifica discrepancias
│       ├── XTPTableExtractor.py         ← Extrae tabla estructurada (regex/pandas)
│       ├── XTPPRLinkerAgent.py          ← Vincula discrepancias a GitHub PRs
│       ├── XTPGeneratorAgent.py         ← Genera Program B + Bin2Bin CSV
│       └── XTPDeliveryAgent.py          ← Escribe archivos resultantes a disco
├── DocumentosXTP/              ← Markdown del manual XTP (alimenta el RAG)
└── Programas/                  ← Directorio de salida de los archivos generados
```

## 🏗️ Arquitectura Agéntica — XTP Analysis Pipeline

El pipeline de análisis es un **StateGraph secuencial de 6 nodos** que toma dos commits SHA de GitHub y produce una tabla de discrepancias enriquecida con referencias a Pull Requests.

```mermaid
flowchart TD
    START([INICIO])
    fetch_programs["fetch_programs\nXTPGitCommitAgent\nout: program_a, program_b, diff"]
    generate_diff["generate_diff\nXTPProgramDiffAgent + RAG\nout: response_xtp_diff"]
    analize_bin2bin["analize_bin2bin\nXTPBin2BinMatrixAgent + RAG\nout: response_bin2bin"]
    justify_mismatches["justify_mismatches\nXTPMismatchJustificationAgent\nout: justification_table"]
    extract_table["extract_justification_table\nXTPTableExtractor - regex/pandas\nout: mismatch_df_json"]
    link_prs["link_prs_to_justifications\nXTPPRLinkerAgent\nout: pr_links_json, pr_summary_md"]
    END([FIN])

    START --> fetch_programs
    fetch_programs --> generate_diff
    generate_diff --> analize_bin2bin
    analize_bin2bin --> justify_mismatches
    justify_mismatches --> extract_table
    extract_table --> link_prs
    link_prs --> END
```

### Descripción de los nodos

| Nodo | Responsabilidad | Clave de estado (output) |
|---|---|---|
| `fetch_programs` | Descarga Program A y Program B desde GitHub por SHA; calcula `unified_diff` | `program_a`, `program_b`, `diff` |
| `generate_diff` | Analiza semánticamente las diferencias entre programas con RAG | `response_xtp_diff` |
| `analize_bin2bin` | Analiza la matriz Bin2Bin CSV para detectar discrepancias con RAG | `response_bin2bin` |
| `justify_mismatches` | Justifica cada discrepancia cruzando diff + análisis Bin2Bin | `justification_table` |
| `extract_justification_table` | Extrae la tabla estructurada vía regex/pandas | `mismatch_df_json` |
| `link_prs_to_justifications` | Vincula cada discrepancia a PRs cerrados de GitHub | `pr_links_json`, `pr_summary_md` |

## 🔗 Sub-agente XTPPRLinkerAgent — Detalle Interno

El último nodo contiene internamente un **pipeline de dos fases asíncronas** que usa GitHub MCP (Model Context Protocol) para descubrir PRs y luego los asocia a cada fila del DataFrame.

```mermaid
flowchart TD
    LINKER["XTPPRLinkerAgent.link"]
    DISC["XTPPRDiscoveryAgent\nPhase 1 - async"]
    MCP["GitHub MCP\nlist_pull_requests state=closed"]
    CAT["PR Catalogue\nJSON array"]
    MATCH["XTPPRMatcherAgent\nPhase 2 - async por fila\nsin tools - todo en prompt"]
    ENRICH["DataFrame enriquecido\npr_numbers, pr_titles, pr_links"]
    MD["pr_summary_md\nMarkdown con hipervinculos a PRs"]

    LINKER --> DISC
    DISC --> MCP --> CAT
    CAT --> MATCH
    MATCH --> ENRICH --> MD
```

## 🏗️ Arquitectura Agéntica — XTP Generation Pipeline

El pipeline de **generación** es independiente del de análisis. Toma un programa XTP base y genera automáticamente un Programa B con delta paramétrico y su matriz Bin2Bin.

```mermaid
flowchart TD
    START([INICIO])
    generate["generate\nXTPGeneratorAgent - ReAct + RAG\ntools: select_random_xtp_delta, generate_bin2bin_csv, find_xtp_documents\nout: generator_output"]
    deliver["deliver\nXTPDeliveryAgent\nout: Program_A.xtp, Program_B.xtp, Bin2Bin_Matrix.csv"]
    END([FIN])

    START --> generate
    generate --> deliver
    deliver --> END
```

### Herramientas del XTPGeneratorAgent

| Herramienta | Propósito |
|---|---|
| `select_random_xtp_delta` | Elige un delta paramétrico aleatorio (bins, límites, parámetros) |
| `generate_bin2bin_csv` | Genera la matriz Bin2Bin en formato CSV a partir de los programas |
| `find_xtp_documents` | Consulta el manual XTP via RAG para validar sintaxis y parámetros |

## 📚 Uso de RAG — XTP Analyzer

El RAG del XTP Analyzer usa el **manual XTP en formato Markdown** como fuente de conocimiento, chunkeado por secciones.

```mermaid
flowchart LR
    MDs["DocumentosXTP/\nMarkdown - manual XTP"]
    MD_PROC["MarkdownProcessor\nchunks por seccion"]
    VS["ChromaDB\ncollection: XTP_Manual\nXTPRagCore"]
    TOOL1["Tool: find_xtp_documents\nXTPGeneratorAgent"]
    TOOL2["Tool: find_xtp_documents\nXTPBin2BinMatrixAgent"]
    AGENT1["XTPGeneratorAgent ReAct"]
    AGENT2["XTPBin2BinMatrixAgent ReAct"]

    MDs --> MD_PROC --> VS
    AGENT1 -- sintaxis XTP --> TOOL1 --> VS
    AGENT2 -- limites y bins --> TOOL2 --> VS
```

### Comparación RAG: Clean Code vs XTP

| Aspecto | Clean Code Reviewer | XTP Analyzer |
|---|---|---|
| **Fuente** | PDFs de buenas prácticas | Markdown del manual XTP |
| **Procesador** | `PDFProcessor` (chunks por capítulo) | `MarkdownProcessor` (chunks por sección) |
| **Colección ChromaDB** | `default` | `XTP_Manual` |
| **Nodos que lo usan** | `read_file` (1 nodo) | `generate_diff` + `analize_bin2bin` + `generate` (3 nodos) |
| **Output** | Campo `ragReference` (APA) en cada finding | Contexto directo en el análisis |

## 🔭 Observabilidad con Langfuse — XTP Analyzer

Ambos pipelines XTP usan el patrón **monkey-patch** sobre `compiled.invoke` para inyectar Langfuse de forma transparente sin modificar el código de los nodos.

```mermaid
flowchart TD
    A["build_analysis_graph"]
    B["Monkey-patch compiled.invoke\na _invoke_with_langfuse"]
    C["get_callback\nsession_id = sha_a..sha_b\nhash MD5 a trace_id"]
    D["trace_name_context XTPAnalyser"]
    E["Cada sub-agente\nget_callback + trace_name_context"]
    F["Langfuse UI\ntrazas bajo XTPAnalyser"]

    A --> B --> C --> D --> E --> F
```

### Comparación de estrategias Langfuse

| Aspecto | Clean Code Reviewer | XTP Analyzer |
|---|---|---|
| **Session ID** | Ruta del archivo | `sha_a[:8]..sha_b[:8]` |
| **Inyección** | `run()` llama `get_callback()` explícitamente | Monkey-patch en `build_*_graph()` |
| **Trace name** | `CleanCodeReviewer` | `XTPAnalyser` |
| **Trazabilidad** | Mismo trace_id por archivo | Mismo trace_id por par de SHAs |

## 🚀 Demo — Ejecutar el XTP Analysis Pipeline

In [1]:
# Ejemplo de ejecución del XTP Analysis Pipeline
# Requiere: GITHUB_PERSONAL_ACCESS_TOKEN, LLM_API_KEY y LANGFUSE_* en el .env
# El logger 'log' y el dropdown ya están disponibles desde la celda de setup.

from XTPAnalyser.AnalysisGraph import build_analysis_graph

# Reusar el logger del dropdown; crear instancia derivada para este pipeline
import logging
xtp_log = log
app = build_analysis_graph(logger=xtp_log)

# Reemplaza con los SHAs reales de los commits a comparar
SHA_A = "abc12345"
SHA_B = "def67890"
BIN2BIN_CSV = "Programas/Bin2Bin_Matrix.csv"  # ruta al archivo CSV generado

xtp_log._logger.info("Iniciando analisis XTP: %s .. %s", SHA_A[:8], SHA_B[:8])
result_xtp = app.invoke({
    "sha_a": SHA_A,
    "sha_b": SHA_B,
    "bin2bin_file": BIN2BIN_CSV,
    "log": xtp_log,
})

xtp_log._logger.info("=== Justification Table ===")
xtp_log._logger.info("%s", result_xtp.get("justification_table", "(no output)"))

d:\Repositorios\prueba_langchain\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


NameError: name 'log' is not defined

In [ ]:
# Visualizar la tabla de discrepancias enriquecida con PRs
import pandas as pd
import io
from IPython.display import display

pr_links_json = result_xtp.get("pr_links_json", "")
if pr_links_json:
    df = pd.read_json(io.StringIO(pr_links_json))
    log._logger.info("DataFrame de discrepancias enriquecido con %d filas", len(df))
    display(df)
else:
    log._logger.warning("No PR linking data available.")

pr_md = result_xtp.get("pr_summary_md", "")
if pr_md:
    log._logger.info("=== PR Summary Markdown ===")
    log._logger.debug("%s", pr_md)
else:
    log._logger.warning("Sin PR summary markdown.")

## 🚀 Demo — Generar un par de programas XTP

In [ ]:
# Ejemplo de ejecución del XTP Generation Pipeline
# Genera Program_B.xtp aplicando un delta parametrico aleatorio a Program_A.xtp
# El logger 'log' y el dropdown ya están disponibles desde la celda de setup.

from XTPAnalyser.graph import build_graph

gen_log = log  # mismo logger — nivel controlado por el dropdown
app_gen = build_graph(logger=gen_log)

# Lee el programa XTP base (Program A)
INPUT_XTP = "Programas/Program_A.xtp"
with open(INPUT_XTP, "r", encoding="utf-8") as f:
    input_program = f.read()

gen_log._logger.info("Generando Program B desde: %s", INPUT_XTP)
result_gen = app_gen.invoke({
    "input_program": input_program,
    "output_folder": "Programas",
    "log": gen_log,
})

gen_log._logger.info("Resultado entrega: %s", result_gen.get("delivery_result", "(sin resultado)"))